# Vole outbreak prepro

In [26]:
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import subprocess

In [27]:
# Directories to input files
root_input_vole_dir = f"data/input/vole" 
# root_input_dir = f"{root_input_vole_dir}/outbreak-{outbreak_num}"
# root_output_dir = f"data/output/vole/outbreak-{outbreak_num}"
gemeente_shp_dir = "data/input/gemeentes/corrected/Gemeentes2013TrMr.shp"

# farm_shp_dir = f"{root_input_dir}/grid.geojson"
# traj_csv_dir = f"{root_input_dir}/trajectory.csv"

# # Directories to output_files
# preprocessed_farm_shp_dir = f"{root_output_dir}/preprocessed_farm.shp"
# iv_table_dir = f"{root_output_dir}/iv_table.csv"

## Function definitions

In [28]:
# Function to transform the crs of farm grid to the crs of the gemeente
def transform_farm_grid_crs(
  farm_gdf,
  gemeente_gdf,
  transformed_farm_shp_dir = "output/farm_transformed_crs.shp",
  save_file=False
):
  if farm_gdf.crs != gemeente_gdf.crs:
    farm_gdf = farm_gdf.to_crs(gemeente_gdf.crs)

  if save_file == True:
    farm_gdf.to_file(transformed_farm_shp_dir)

  return farm_gdf


# Function to transform each farm polygon to centroids
def convert_farm_to_centroids(
  farm_gdf,
  farm_centroid_dir = "output/farm_centroid.shp",
  save_file = False
):

  farm_gdf["geometry"] = farm_gdf["geometry"].centroid

  if save_file == True:
    farm_gdf.to_file(farm_centroid_dir)

  return farm_gdf


# Function to add to each farm the gemeente ID of the gemeente they belong to
def add_gemeente_ids_to_farm_centroids(
  farm_gdf,
  gemeente_gdf,
  output_farm_dir = "output/farm_with_gm_id.shp",
  save_file = False
):
  # Spatial join to match grid points with gemeente geometries
  joined_gdf = gpd.sjoin(left_df= farm_gdf, right_df=gemeente_gdf[["OBJECTID", "geometry"]], how="left", predicate="within")
 
  # Assign the gemeente_id from the join result
  farm_gdf["gm_id"] = joined_gdf["OBJECTID"]

  # Saving file
  if save_file:
    farm_gdf.to_file(output_farm_dir)

  return farm_gdf

# Function to add to each farm the gemeente ID of the gemeente they belong to
def add_gemeente_names_to_farm_centroids(
  farm_gdf,
  gemeente_gdf,
  output_farm_dir = "output/farm_with_gm_id.shp",
  save_file = False
):
  # Spatial join to match grid points with gemeente geometries
  joined_gdf = gpd.sjoin(left_df= farm_gdf, right_df=gemeente_gdf[["gemeente", "geometry"]], how="left", predicate="within")
 
  # Assign the gemeente_id from the join result
  farm_gdf["name"] = joined_gdf["gemeente"]

  # Saving file
  if save_file:
    farm_gdf.to_file(output_farm_dir)

  return farm_gdf

def discard_farms_outside_nl(
  farm_gdf
):
  # Discarding farms that have NA for gm_id. This means these farms
  # are outside the NL
  farm_gdf = farm_gdf[farm_gdf["name"].notna()].copy()

  # Convert to integer type if no missing values are expected
  farm_gdf["name"] = farm_gdf["name"].astype("str")
  farm_gdf["node"] = farm_gdf["node"].astype("int")

  farm_gdf = farm_gdf[farm_gdf["node"] != 19271]

  return(farm_gdf)

In [29]:
# Function to transform the iv table so that each row represents the time and the
# columns are the
def create_iv_tables(
    traj_csv_dir,
    output_iv_table_dir="output/env_table.csv",
    preprocessed_farm_dir="output/preprocessed_farm.shp"
):

    trajectory_data = pd.read_csv(traj_csv_dir)
    grid_df = gpd.read_file(preprocessed_farm_dir)

    # Pivot the data on time column so each time is on the row and the columns are the nodes' IV values
    iv_table = trajectory_data.pivot(index="time", columns="node", values="ENV").reset_index()

    # Removing the column index name
    iv_table.columns.name = None

    # Deleting columns corresponding to IV values for nodes that are not in preprocessed grid.
    # The nodes not present in preprocessed grid were outside of the country
    iv_table = iv_table[iv_table.columns.intersection(grid_df["node"].values)]

    iv_table.to_csv(output_iv_table_dir, index=False)

### Script to run preprocessing

In [ ]:
for dir in os.listdir(root_input_vole_dir):
    if dir.startswith('outbreak-'):
        outbreak_num = dir.split('-')[1]

    else:
       continue
    
    root_input_dir = f"{root_input_vole_dir}/outbreak-{outbreak_num}"
    root_output_dir = f"data/output/vole/outbreak_{outbreak_num}"

    # Directories to input files
    farm_shp_dir = f"{root_input_dir}/grid.geojson"
    traj_csv_dir = f"{root_input_dir}/trajectory.csv"

    # We need to unzip the traj csv file
    if "trajectory.csv" not in os.listdir(root_input_dir):
       traj_zip_dir = f"{root_input_dir}/trajectory.csv.gz"
       subprocess.run(['gunzip', traj_zip_dir])

    # Directories to output_files
    preprocessed_farm_shp_dir = f"{root_output_dir}/vole_loc.shp"
    iv_table_dir = f"{root_output_dir}/env_table.csv"

    # Script to run preprocessing
    og_farm_gdf = gpd.read_file(farm_shp_dir) 
    gemeente_gdf = gpd.read_file(gemeente_shp_dir)

    transformed_farm_gdf = transform_farm_grid_crs(og_farm_gdf, gemeente_gdf)
    farm_centroids_gdf = convert_farm_to_centroids(transformed_farm_gdf)
    farm_with_gm_gdf = add_gemeente_names_to_farm_centroids(farm_centroids_gdf, gemeente_gdf)
    farm_gm_nl_only_gdf = discard_farms_outside_nl(farm_with_gm_gdf)

    if not os.path.isdir(root_output_dir):
      os.makedirs(root_output_dir)

    farm_gm_nl_only_gdf.to_file(preprocessed_farm_shp_dir)

    print(f"Created output file for {outbreak_num}")
    create_iv_tables(traj_csv_dir, iv_table_dir,preprocessed_farm_shp_dir)

    df = pd.read_csv(iv_table_dir)
    print(df.max().max())

print("Preprocessing complete")

Created output file for 917448
531.652565973849
Created output file for 654502
1593.60579985362
Created output file for 717926
229.773132713687
Created output file for 403228
634.956226941094
Created output file for 848178
560.4493276077
Created output file for 282382
603.139708938952
Preprocessing complete


In [41]:
outbreak_num = 848178
root_output_dir = f"data/output/vole/outbreak-{outbreak_num}"
iv_table_dir = f"{root_output_dir}/env_table.csv"

df = pd.read_csv(iv_table_dir)

In [49]:
row_numbers = df.reset_index().index[(df > 0).any(axis=1)].tolist()
row_numbers

/var/folders/vr/8r4ct0755_nfft1p7z7ldhym0000gp/T/ipykernel_2025/413677364.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  row_numbers = df.reset_index().index[(df > 0).any(axis=1)].tolist()


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
